In [55]:
%pip install torch torchvision torchaudio

  Using cached filelock-3.29.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/122.0 MB ? eta -:--:--
    --------------------------------------- 1.6/122.0 MB 10.5 MB/s eta 0:00:12
   - -------------------------------------- 3.1/122.0 MB 8.4 MB/s eta 0:00:15
   - -------------------------------------- 5.0/122.0 MB 8.6 MB/s eta 0:00:14
   -- ------------------------------------- 7.1/122.0 MB 9.3 MB/s eta 0:00:13
   -- ------------------------------------- 8.9/122.0 MB 8.8 MB/s eta 0:00:13
   --- ------------------------------------ 11.0/122.0 MB 8.9 MB/s eta 0:00:13
   ---- ----------------------------------- 13.1/122.0 MB 9.1 MB/s eta 0:00:12
   ----- ---------------------------------- 15.7/122.0 MB 9.6 MB/s eta 0:00:12
   ----- ---------------------------------- 1

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [80]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader


In [81]:
# Download the preprocessed data
X_train = pd.read_csv("../clean_data/X_train.csv")
X_test = pd.read_csv("../clean_data/X_test.csv")
y_train = pd.read_csv("../clean_data/y_train.csv").squeeze()
y_test = pd.read_csv("../clean_data/y_test.csv").squeeze()


In [82]:
# Encodage de la variaable mois en variables sin et cos pour capturer la cyclicité
X_train["Month_sin"] = np.sin(2 * np.pi * X_train["Month"] / 12)
X_train["Month_cos"] = np.cos(2 * np.pi * X_train["Month"] / 12)

X_test["Month_sin"] = np.sin(2 * np.pi * X_test["Month"] / 12)
X_test["Month_cos"] = np.cos(2 * np.pi * X_test["Month"] / 12)

# Encodage de la variable "Day" en variables sin et cos pour capturer la cyclicité
X_train["Day_sin"] = np.sin(2 * np.pi * X_train["Day"] / 31)
X_train["Day_cos"] = np.cos(2 * np.pi * X_train["Day"] / 31)

X_test["Day_sin"] = np.sin(2 * np.pi * X_test["Day"] / 31)
X_test["Day_cos"] = np.cos(2 * np.pi * X_test["Day"] / 31)

# Sppression de la variable "Month" et "Day" après l'encodage
X_train = X_train.drop(columns=["Month", "Day"])
X_test = X_test.drop(columns=["Month", "Day"])

X_train.head()

,Transaction_Amount (in Million),Distance_From_Home,Account_Balance (in Million),Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,...,Customer_Home_Location_Islamabad,Customer_Home_Location_Karachi,Customer_Home_Location_Lahore,Customer_Home_Location_Multan,Card_Type_Credit,Card_Type_Debit,Month_sin,Month_cos,Day_sin,Day_cos
0,1.0,212.0,6.0,6.0,1.0,2.0,5.0,1,1,2.0,...,0,0,1,0,1,0,0.866025,-0.5,-0.998717,-0.050649
1,2.0,170.0,32.0,6.0,22.0,2.0,5.0,1,0,0.0,...,0,0,0,0,0,1,0.866025,-0.5,0.848644,0.528964
2,4.0,49.0,39.0,2.0,3.0,4.0,7.0,0,1,1.0,...,0,0,0,0,0,1,0.866025,-0.5,-0.101168,-0.994869
3,4.0,212.0,10.0,2.0,8.0,3.0,3.0,1,1,1.0,...,0,1,0,0,1,0,0.866025,-0.5,-0.988468,0.151428
4,9.0,507.0,30.0,1.0,21.0,5.0,3.0,0,1,0.0,...,0,0,1,0,0,1,0.866025,-0.5,0.988468,0.151428


In [83]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [84]:
# Conversion des données en tensor
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)


In [85]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [86]:
print(len(train_dataset))
print(train_dataset[0])

40000
(tensor([-1.5508, -0.5126, -1.3996,  1.0008, -1.6657, -0.7002, -0.0070,  1.0000,
         0.9982,  1.2334, -0.9942, -1.0047, -0.7944, -0.1990,  0.0000, -0.7104,
        -0.7089,  1.4245, -0.4496, -0.4465,  2.2531, -0.4473, -0.4451, -0.4508,
        -0.3353, -0.3300, -0.3343, -0.3356, -0.3327,  2.9929, -0.3326, -0.3309,
        -0.3343, -0.3335, -0.4980, -0.4992, -0.5005,  1.9908, -0.5000,  1.0024,
        -1.0024,  0.3249, -1.3425, -1.4128, -0.0276]), tensor(0.))


In [87]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [88]:
print(len(train_loader))
print(len(test_loader))

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

625
157
torch.Size([64, 45])
torch.Size([64])


In [ ]:
class MLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(45, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):

        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.output(x)

        return x
    

In [91]:
model = MLP()
print(model)

MLP(
  (fc1): Linear(in_features=45, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=16, bias=True)
  (output): Linear(in_features=16, out_features=1, bias=True)
)


In [92]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 50

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        y_batch = y_batch.unsqueeze(1)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")


Epoch [1/20] - Loss: 0.1563
Epoch [2/20] - Loss: 0.1541
Epoch [3/20] - Loss: 0.1528
Epoch [4/20] - Loss: 0.1509
Epoch [5/20] - Loss: 0.1491
Epoch [6/20] - Loss: 0.1483
Epoch [7/20] - Loss: 0.1461
Epoch [8/20] - Loss: 0.1443
Epoch [9/20] - Loss: 0.1430
Epoch [10/20] - Loss: 0.1417
Epoch [11/20] - Loss: 0.1402
Epoch [12/20] - Loss: 0.1392
Epoch [13/20] - Loss: 0.1381
Epoch [14/20] - Loss: 0.1374
Epoch [15/20] - Loss: 0.1356
Epoch [16/20] - Loss: 0.1354
Epoch [17/20] - Loss: 0.1336
Epoch [18/20] - Loss: 0.1331
Epoch [19/20] - Loss: 0.1312
Epoch [20/20] - Loss: 0.1298
